# ANN Implementation for Customer Churn Prediction

In [228]:
import tensorflow as tf

In [229]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

In [230]:
import os

In [231]:
# Load preprocessed train and test data
# Data should already be scaled and encoded before loading
import pandas as pd

train_path = os.path.join("data", "train_data.csv")
test_path = os.path.join("data", "test_data.csv")
    
train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

X_train = train_data.drop("Exited", axis=1)
y_train = train_data["Exited"]

X_test = test_data.drop("Exited", axis=1)
y_test = test_data["Exited"]

In [232]:
train_data.tail()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Male,Exited
7995,1.207474,1.435808,1.039728,-0.102301,-0.916688,0.649203,0.974817,-0.539860,1.001501,-0.579467,-0.576388,0.913248,0
7996,0.314989,1.816097,-1.389442,-1.218471,-0.916688,0.649203,0.974817,-1.733882,1.001501,-0.579467,-0.576388,-1.094993,0
7997,0.865009,-0.085351,-1.389442,-1.218471,2.533560,-1.540351,-1.025834,-0.142765,1.001501,-0.579467,-0.576388,-1.094993,1
7998,0.159323,0.390011,1.039728,1.827259,-0.916688,0.649203,-1.025834,-0.050826,1.001501,-0.579467,-0.576388,0.913248,1
7999,0.470655,1.150590,-1.389442,1.149720,-0.916688,0.649203,0.974817,-0.814568,-0.998501,1.725723,-0.576388,0.913248,0


In [233]:
train_data.isna().sum()

CreditScore          0
Age                  0
Tenure               0
Balance              0
NumOfProducts        0
HasCrCard            0
IsActiveMember       0
EstimatedSalary      0
Geography_France     0
Geography_Germany    0
Geography_Spain      0
Gender_Male          0
Exited               0
dtype: int64

In [234]:
(X_train.shape[1],)

(12,)

In [235]:
y_train.unique()

array([0, 1])

In [236]:
y_train.shape

(8000,)

In [237]:
y_train.value_counts()

Exited
0    6356
1    1644
Name: count, dtype: int64

In [238]:
6356 + 1644

8000

In [239]:
# Diagnostic checks before training
import numpy as np

def check_data(X, y, name="data"):
    print(f"Checking {name}: X shape={getattr(X,'shape',None)}, y shape={getattr(y,'shape',None)}")
    # Convert to numpy for checks
    X_arr = np.array(X)
    y_arr = np.array(y)
    print("  dtype X:", X_arr.dtype, "dtype y:", y_arr.dtype)
    print("  any NaN X:", np.isnan(X_arr).any(), "any NaN y:", np.isnan(y_arr).any())
    print("  any Inf X:", np.isinf(X_arr).any(), "any Inf y:", np.isinf(y_arr).any())
    print("  X min/max:", np.nanmin(X_arr), np.nanmax(X_arr))
    print("  y unique values:", np.unique(y_arr)[:10])

check_data(X_train, y_train, "train")
check_data(X_test, y_test, "test")

Checking train: X shape=(8000, 12), y shape=(8000,)
  dtype X: float64 dtype y: int64
  any NaN X: False any NaN y: False
  any Inf X: False any Inf y: False
  X min/max: -3.1304179222804187 5.04856027947936
  y unique values: [0 1]
Checking test: X shape=(2000, 12), y shape=(2000,)
  dtype X: float64 dtype y: int64
  any NaN X: False any NaN y: False
  any Inf X: False any Inf y: False
  X min/max: -3.1304179222804187 5.04856027947936
  y unique values: [0 1]


In [240]:
# Building the ANN model
# Architecture:
# - Input Layer: Accepts features with shape (X_train.shape[1],)
# - Hidden Layer 1: 64 neurons with ReLU activation for non-linearity
# - Hidden Layer 2: 32 neurons with ReLU activation (progressive dimension reduction)
# - Output Layer: 1 neuron with sigmoid activation for binary classification (0-1 probability)
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # HL1
    Dense(32, activation='relu'), # HL2
    Dense(1, activation='sigmoid') # Output
    ]
)

/workspaces/ANN-Customer-Churn/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [241]:
model.summary()

Model: "sequential_10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_30 (Dense)                │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_32 (Dense)                │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [242]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy

In [243]:
# Configure optimizer and loss function
# Adam optimizer: Adaptive learning rate optimization algorithm
# Learning rate 0.01: Controls the step size during gradient descent
# Binary Crossentropy: Standard loss function for binary classification problems
opt = Adam(learning_rate=0.01)
loss = BinaryCrossentropy()

In [244]:
model.compile(optimizer=opt, loss='binary_crossentropy', metrics=['accuracy'])

In [245]:
# Configure TensorBoard callback for training visualization
# TensorBoard logs training metrics, loss curves, and model graphs
# histogram_freq=1: Records weight/bias distributions at each epoch
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

log_dir = 'logs/fit/' + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)

In [246]:
# Configure Early Stopping to prevent overfitting
# Monitors validation loss and stops training when it stops improving
# patience=10: Waits for 10 epochs without improvement before stopping
# restore_best_weights=True: Reverts model to the best epoch's weights
early_stopping_callback = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)

In [247]:
# Train the model with validation
# epochs=100: Maximum training iterations (may stop early due to EarlyStopping)
# validation_data: Evaluates model on unseen test data after each epoch
# Callbacks: TensorBoard for visualization, EarlyStopping for regularization
history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
 37/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7660 - loss: 0.4991    

250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8338 - loss: 0.3954 - val_accuracy: 0.8580 - val_loss: 0.3556
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8536 - loss: 0.3534 - val_accuracy: 0.8570 - val_loss: 0.3482
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8585 - loss: 0.3472 - val_accuracy: 0.8510 - val_loss: 0.3549
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8612 - loss: 0.3422 - val_accuracy: 0.8615 - val_loss: 0.3346
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8624 - loss: 0.3393 - val_accuracy: 0.8600 - val_loss: 0.3411
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8616 - loss: 0.3374 - val_accuracy: 0.8605 - val_loss: 0.3480
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8619 - loss: 0.3365 - val_accuracy: 0.8530 - val_loss: 0.3467
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8637 - loss: 0.3338 - val_accuracy: 0.8605

In [248]:
model_save_dir = "model_artifacts"
os.makedirs(model_save_dir, exist_ok=True)

model_path = os.path.join(model_save_dir, "customer_churn_model.keras")
model.save(model_path)

h5_model = os.path.join(model_save_dir, "customer_churn_model.h5")
model.save(h5_model)

In [249]:
# Load TensorBoard extension for in-notebook visualization
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [250]:
# Launch TensorBoard to visualize training metrics
# Displays loss curves, accuracy trends, and model architecture
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 3933), started 0:51:13 ago. (Use '!kill 3933' to kill it.)